In [1]:
import os
import requests
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import mlflow
import shutil

RETRAIN = False
DATA_PATH = "../../02_data/mnist"
NUM_CLIENTS = 3

mlflow.set_tracking_uri("http://localhost:5100")
MLFLOW_EXPERIMENT_NAME = "edge-fl-experiment"

I0000 00:00:1791551829.852731    3042 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
#!docker pull ghcr.io/bencefarkas27/aggregator:test-auth
!docker run --name aggregator_test -d --rm -p 5100:5000 -p 9100:8000  ghcr.io/bencefarkas27/aggregator:test

### Healt endpoint check

In [2]:
response = requests.get("http://localhost:9100/health")
print(f"Health check response of aggregator: {response.status_code} - {response.text}")

Health check response of aggregator: 200 - 


In [3]:
run_ids = []

def create_dummy_model(input_shape):
    model = keras.Sequential([
        keras.layers.InputLayer(shape=input_shape),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_client(client_id):
    # Load client data
    x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
    y_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_labels.npy")
    # Create and train the model
    model = create_dummy_model(input_shape=x_train.shape[1:])
    model.fit(x_train, y_train, epochs=3)
    # Save the trained model weights
    model.save_weights(f"dummy_models/client_{client_id}.weights.h5")

# def mlflow_log_client_model(client_id):
#     weights_path = f"dummy_models/client_{client_id}.weights.h5"
#     x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
#     input_shape = x_train.shape[1:]
#     model = create_dummy_model(input_shape=input_shape)
#     model.load_weights(weights_path)
#     model.compile(optimizer='adam',
#                   loss='sparse_categorical_crossentropy',
#                   metrics=['accuracy'])
#     with mlflow.start_run(run_name=f"client_{client_id}_run") as run:
#         # Log the model to MLflow
#         mlflow.tensorflow.log_model(model, f"client_{client_id}_model")
#         run_ids.append(run.info.run_id)


# def mlflow_log_client_model(client_id):
#     response = requests.get(f"http://localhost:8555/client_{client_id}.weights.h5")
#     response.raise_for_status()  # raises an exception on 4xx/5xx instead of silently continuing

#     weights_path = f"client_{client_id}.weights.h5"
#     with open(weights_path, "wb") as f:
#         f.write(response.content)
#     print(f"Client {client_id} weights downloaded.")
#     input_shape = os.environ.get("MLFLOW_INPUT_SHAPE", (28, 28))
#     model = create_dummy_model(input_shape=input_shape)
#     model.load_weights(weights_path)
#     model.compile(optimizer='adam',
#                   loss='sparse_categorical_crossentropy',
#                   metrics=['accuracy'])
#     with mlflow.start_run(run_name=f"client_{client_id}_run") as run:
#         # Log the model to MLflow
#         mlflow.tensorflow.log_model(model, f"client_{client_id}_model")
#         return run.info.run_id

# def mlflow_load_client_model(as_user, experiment_name, mlflow_client_number):
#     as_user("admin")
#     mlflow.set_experiment(experiment_name)
#     for client_id in range(mlflow_client_number):
#         mlflow_log_client_model(client_id)

def post_model_to_aggregator(model_path, model_name, model_id):
    # 1. Download the model weights from the Fured cloud storage
    response = requests.get(f"http://localhost:8555/{model_name}.weights.h5")
    response.raise_for_status()  # raises an exception on 4xx/5xx instead of silently continuing
    os.makedirs(model_path, exist_ok=True)
    # 2. Save the weights to a local file
    weights_path = f"{model_path}/model.weights.h5"
    with open(weights_path, "wb") as f:
        f.write(response.content)
    print(f"Model {model_name} weights downloaded.")
    # 3. Reconstruct the model architecture and load the weights
    input_shape = os.environ.get("MLFLOW_INPUT_SHAPE", (28, 28))
    model = create_dummy_model(input_shape=input_shape)
    model.load_weights(weights_path)
    model.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])
    # 4. Save the model structure to a JSON file
    model_structure = model.to_json()
    with open(f"{model_path}/model.json", "w") as json_file:
        json.dump(json.loads(model_structure), json_file)
    # 5. Create a zip which contains both the model structure and the weights
    shutil.make_archive("model", "zip", model_path)
    with open("model.zip", "rb") as f:
        model_data = f.read()
    # 6. Post the model zip to the aggregator
    response = requests.post(
        f"http://localhost:9100/post_model",
        params={"model_id": model_id},
        data=model_data
    )
    if response.status_code != 200:
        raise Exception(f"Failed to forward model to aggregator: {response.text}")


In [4]:
for client_id in range(NUM_CLIENTS):
    if RETRAIN:
        train_client(client_id)
    post_model_to_aggregator(f"dummy_models/client_{client_id}", f"client_{client_id}", f"model_{client_id}")

Model client_0 weights downloaded.


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model client_1 weights downloaded.


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_2', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model client_2 weights downloaded.


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_4', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### Calling do_aggregate with fedavg

In [5]:
aggregation_config = {
    "model_sources": [
        f"client_{client_id}" for client_id in range(NUM_CLIENTS)
    ],
    "model_ids": [
        f"model_{client_id}" for client_id in range(NUM_CLIENTS)
    ],
    "local_data_sizes": [
        len(np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy"))
        for client_id in range(NUM_CLIENTS)
    ],
    "aggregation_algorithm": "fedavg",
    "forward_model_to": "http://top_aggregator:8000",
}

response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)

In [6]:
print(response.json())

{'aggregated_model_host_external': 'http://localhost:5100', 'aggregated_model_host_organizational': 'http://aggregator-top:5000', 'aggregated_model_uri': 'models:/m-e964b3054da24454a163fabf67277afe', 'aggregated_model_username': 'model_user', 'aggregated_model_password': 'hiPMMENYuzajkO_vuq9owKnV3ORxRSapm7GgpNw9BAU'}


### Pulling from mlflow

In [7]:
# Run these for client model training and logging to MLflow if no volume mounted client models exist for mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "admin"
with open("../../04_deploy/aggregator/cfg/mlflow_admin.json", "r") as f:
                admin_config = json.load(f)
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["admin_password"]
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1791551460646, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791551460646, lifecycle_stage='active', name='edge-fl-experiment', tags={}, trace_location=None, workspace='default'>

In [8]:
def get_model_ids():
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    models = mlflow.search_logged_models(
        experiment_ids=[experiment.experiment_id],
        order_by=[{"field_name": "creation_time", "ascending": False}],
        output_format="list"
    )
    model_ids = [model.model_id for model in models]
    return model_ids
print(get_model_ids())

['m-e964b3054da24454a163fabf67277afe']


In [9]:
# Testing the client models vs the aggregated model
# Load test data
x_test = np.load(f"{DATA_PATH}/test_images.npy")
y_test = np.load(f"{DATA_PATH}/test_labels.npy")

In [10]:
# Pull the aggregated model from mlflow
try:
    model_ids = get_model_ids()
    latest_model_id = model_ids[0]
    aggregated_model = mlflow.keras.load_model(f"models:/{latest_model_id}")
    aggregated_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print(f"Loaded latest aggregated model: {latest_model_id}")

    aggregated_loss, aggregated_accuracy = aggregated_model.evaluate(x_test, y_test, verbose=0)
    print(f"Aggregated Model - Loss: {aggregated_loss}, Accuracy: {aggregated_accuracy}")
except Exception as e:
    print(f"Failed to load model: {e}")


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Loaded latest aggregated model: m-e964b3054da24454a163fabf67277afe
Aggregated Model - Loss: 11.153631210327148, Accuracy: 0.20409999787807465


In [11]:
client_losses = []
client_accuracies = []
for client_id in range(NUM_CLIENTS):
    # Load the client model
    client_model = create_dummy_model(input_shape=x_test.shape[1:])
    client_model.load_weights(f"dummy_models/client_{client_id}/model.weights.h5")
    
    # Evaluate the client model
    client_loss, client_accuracy = client_model.evaluate(x_test, y_test, verbose=0)
    print(f"Client {client_id} Model - Loss: {client_loss}, Accuracy: {client_accuracy}")
    client_losses.append(client_loss)
    client_accuracies.append(client_accuracy)

/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_7', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 0 Model - Loss: 96.55555725097656, Accuracy: 0.4092999994754791


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_8', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 1 Model - Loss: 159.5695037841797, Accuracy: 0.2930000126361847


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_9', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 2 Model - Loss: 222.02365112304688, Accuracy: 0.2921000123023987


In [12]:
print(f"Aggregator's loss is {aggregated_loss} vs clients's losses: {client_losses} and avarage client loss: {np.mean(client_losses)}")
print(f"Aggregator's accuracy is {aggregated_accuracy} vs clients's accuracies: {client_accuracies} and avarage client accuracy: {np.mean(client_accuracies)}")

Aggregator's loss is 11.153631210327148 vs clients's losses: [96.55555725097656, 159.5695037841797, 222.02365112304688] and avarage client loss: 159.38290405273438
Aggregator's accuracy is 0.20409999787807465 vs clients's accuracies: [0.4092999994754791, 0.2930000126361847, 0.2921000123023987] and avarage client accuracy: 0.3314666748046875


In [58]:
# Saving the aggregated model to a file
aggregated_model.save_weights(f"dummy_models/aggregated_model.weights.h5")

In [59]:
!docker stop aggregator_test

Error response from daemon: No such container: aggregator_test
